
<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 60px 40px; border-radius: 16px; font-family: 'Segoe UI', sans-serif; color: #fff; text-align: center; margin-bottom: 10px;">
  <div style="display: inline-block; background: rgba(255,255,255,0.08); border: 1px solid rgba(255,255,255,0.18); border-radius: 50px; padding: 8px 24px; margin-bottom: 20px;">
    <span style="font-size: 13px; letter-spacing: 3px; text-transform: uppercase; color: #a78bfa;">Cognifyz Technologies · Data Analytics Internship</span>
  </div>
  <h1 style="font-size: 2.8em; font-weight: 800; margin: 0 0 12px; letter-spacing: -1px;">
    Price Range vs. Online Delivery<br>&amp; Table Booking
  </h1>
  <p style="font-size: 1.15em; color: #c4b5fd; max-width: 680px; margin: 0 auto 32px; line-height: 1.7;">
    A comprehensive statistical analysis examining the relationship between restaurant price segments
    and the availability of online delivery and table booking services.
  </p>
  <hr style="border: none; border-top: 1px solid rgba(255,255,255,0.15); margin: 30px 0;" />
  <div style="display: flex; justify-content: center; gap: 60px; flex-wrap: wrap; margin-top: 20px;">
    <div>
      <div style="font-size: 12px; letter-spacing: 2px; text-transform: uppercase; color: #a78bfa; margin-bottom: 6px;">Analyst</div>
      <div style="font-size: 1.1em; font-weight: 600;">Karthikeyan Thirunavukkarasu</div>
    </div>
    <div>
      <div style="font-size: 12px; letter-spacing: 2px; text-transform: uppercase; color: #a78bfa; margin-bottom: 6px;">Dataset</div>
      <div style="font-size: 1.1em; font-weight: 600;">Cognifyz Restaurant Dataset</div>
    </div>
    <div>
      <div style="font-size: 12px; letter-spacing: 2px; text-transform: uppercase; color: #a78bfa; margin-bottom: 6px;">Records</div>
      <div style="font-size: 1.1em; font-weight: 600;">9,551 Restaurants</div>
    </div>
    <div>
      <div style="font-size: 12px; letter-spacing: 2px; text-transform: uppercase; color: #a78bfa; margin-bottom: 6px;">Tool</div>
      <div style="font-size: 1.1em; font-weight: 600;">Python · Pandas · Matplotlib · Seaborn</div>
    </div>
  </div>
</div>



<div style="background: #1e1e2e; border-left: 5px solid #7c3aed; border-radius: 10px; padding: 28px 32px; font-family: 'Segoe UI', sans-serif; color: #e2e8f0; margin: 10px 0;">
  <h2 style="color: #a78bfa; font-size: 1.3em; letter-spacing: 1px; text-transform: uppercase; margin: 0 0 16px;">📌 Analytical Objectives</h2>
  <ul style="list-style: none; padding: 0; margin: 0; line-height: 2.2; font-size: 1.02em;">
    <li>&#x2713;&nbsp; <strong>Quantify</strong> the distribution of online delivery availability across all price tiers</li>
    <li>&#x2713;&nbsp; <strong>Quantify</strong> the distribution of table booking availability across all price tiers</li>
    <li>&#x2713;&nbsp; <strong>Determine</strong> whether higher-priced restaurants are more likely to offer these services</li>
    <li>&#x2713;&nbsp; <strong>Validate</strong> statistical significance using Chi-Square tests</li>
    <li>&#x2713;&nbsp; <strong>Surface</strong> actionable business insights from the patterns observed</li>
  </ul>
</div>


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# IMPORTS & CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings("ignore")

# ── Global aesthetics ────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.facecolor": "#0f0c29",
    "axes.facecolor":   "#1a1a2e",
    "axes.edgecolor":   "#3a3a5c",
    "axes.labelcolor":  "#e2e8f0",
    "xtick.color":      "#a0aec0",
    "ytick.color":      "#a0aec0",
    "text.color":       "#e2e8f0",
    "grid.color":       "#2d2d4e",
    "grid.linestyle":   "--",
    "grid.alpha":       0.5,
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   14,
    "axes.labelsize":   12,
    "figure.dpi":       130,
})

PALETTE_DELIVERY = ["#ef4444", "#22d3ee"]   # No / Yes
PALETTE_BOOKING  = ["#f97316", "#4ade80"]
PRICE_COLORS     = ["#6366f1", "#8b5cf6", "#a855f7", "#c084fc"]
PRICE_LABELS     = {1: "Budget (₹)", 2: "Moderate (₹₹)", 3: "Premium (₹₹₹)", 4: "Luxury (₹₹₹₹)"}

# ── Load Data ────────────────────────────────────────────────────────────────
df = pd.read_csv("cognifyz_dataset.csv")
df["Price Label"] = df["Price range"].map(PRICE_LABELS)

print(f"✅  Dataset loaded — {df.shape[0]:,} rows × {df.shape[1]} columns")
df[["Price range", "Has Online delivery", "Has Table booking", "Average Cost for two", "Aggregate rating"]].head(8)



<div style="background: linear-gradient(90deg, #312e81, #1e1b4b); border-radius: 10px; padding: 22px 32px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
  <h2 style="color: #c7d2fe; margin: 0; font-size: 1.5em; letter-spacing: -0.3px;">
    &#9632;&nbsp; Section 1 — Exploratory Data Analysis
  </h2>
  <p style="color: #a5b4fc; margin: 8px 0 0; font-size: 0.97em;">Dataset profiling · class balance · distribution overview</p>
</div>


In [ ]:

# ── 1. Distribution of Price Ranges ─────────────────────────────────────────
price_dist = df["Price range"].value_counts().sort_index()
delivery_dist = df["Has Online delivery"].value_counts()
booking_dist  = df["Has Table booking"].value_counts()

print("=" * 55)
print("  PRICE RANGE DISTRIBUTION")
print("=" * 55)
for k, v in PRICE_LABELS.items():
    pct = price_dist[k] / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"  Tier {k}  {v:<22} {price_dist[k]:>5,}  ({pct:.1f}%) {bar}")

print()
print("=" * 55)
print("  SERVICE AVAILABILITY")
print("=" * 55)
for label, dist in [("Online Delivery", delivery_dist), ("Table Booking", booking_dist)]:
    yes_pct = dist.get("Yes", 0) / len(df) * 100
    no_pct  = dist.get("No",  0) / len(df) * 100
    print(f"  {label:<18} YES: {yes_pct:.1f}%   NO: {no_pct:.1f}%")
print("=" * 55)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.suptitle("Dataset Overview — Price Range & Service Availability",
             fontsize=15, fontweight="bold", color="#e2e8f0", y=1.01)

# Price range bar
ax = axes[0]
bars = ax.bar([PRICE_LABELS[i] for i in sorted(PRICE_LABELS)],
              [price_dist[i] for i in sorted(price_dist.index)],
              color=PRICE_COLORS, edgecolor="#0f0c29", linewidth=1.2, zorder=3)
ax.set_title("Price Range Distribution", fontweight="bold")
ax.set_xlabel("Price Tier"); ax.set_ylabel("Number of Restaurants")
ax.grid(axis="y", zorder=0)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f"{int(bar.get_height()):,}", ha="center", va="bottom",
            fontsize=10, fontweight="bold", color="#e2e8f0")
ax.set_xticklabels([PRICE_LABELS[i] for i in sorted(PRICE_LABELS)], rotation=15, ha="right", fontsize=9)

# Online delivery donut
ax2 = axes[1]
vals = [delivery_dist.get("No",0), delivery_dist.get("Yes",0)]
wedges, texts, autotexts = ax2.pie(vals, labels=["No","Yes"],
    autopct="%1.1f%%", colors=PALETTE_DELIVERY,
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor="#0f0c29", linewidth=2))
for at in autotexts: at.set(fontsize=11, fontweight="bold", color="white")
ax2.set_title("Online Delivery Availability", fontweight="bold")

# Table booking donut
ax3 = axes[2]
vals2 = [booking_dist.get("No",0), booking_dist.get("Yes",0)]
wedges2, texts2, autotexts2 = ax3.pie(vals2, labels=["No","Yes"],
    autopct="%1.1f%%", colors=PALETTE_BOOKING,
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor="#0f0c29", linewidth=2))
for at in autotexts2: at.set(fontsize=11, fontweight="bold", color="white")
ax3.set_title("Table Booking Availability", fontweight="bold")

plt.tight_layout()
plt.savefig("plot_overview.png", bbox_inches="tight", facecolor="#0f0c29")
plt.show()



<div style="background: linear-gradient(90deg, #1a3a1a, #0f2d0f); border-radius: 10px; padding: 22px 32px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
  <h2 style="color: #bbf7d0; margin: 0; font-size: 1.5em;">
    &#9632;&nbsp; Section 2 — Online Delivery vs. Price Range
  </h2>
  <p style="color: #86efac; margin: 8px 0 0; font-size: 0.97em;">Adoption rate per tier · grouped analysis · trend detection</p>
</div>


In [ ]:

# ── Cross-tabulation ─────────────────────────────────────────────────────────
delivery_ct = pd.crosstab(df["Price range"], df["Has Online delivery"])
delivery_pct = delivery_ct.div(delivery_ct.sum(axis=1), axis=0) * 100
delivery_pct.index = [PRICE_LABELS[i] for i in delivery_pct.index]

print("\n  ONLINE DELIVERY — PERCENTAGE BY PRICE TIER")
print("  " + "─" * 52)
print(f"  {'Price Tier':<26} {'No':>8}   {'Yes':>8}   {'YES %':>8}")
print("  " + "─" * 52)
for idx, row in delivery_pct.iterrows():
    bar = "▓" * int(row["Yes"] / 4)
    print(f"  {idx:<26} {row['No']:>7.1f}%  {row['Yes']:>7.1f}%   {bar}")
print("  " + "─" * 52)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Online Delivery Availability by Price Range",
             fontsize=15, fontweight="bold", color="#e2e8f0")

# Grouped bar
ax = axes[0]
x = np.arange(len(delivery_pct))
w = 0.38
b1 = ax.bar(x - w/2, delivery_pct["No"],  width=w, label="No",  color="#ef4444", edgecolor="#0f0c29", zorder=3)
b2 = ax.bar(x + w/2, delivery_pct["Yes"], width=w, label="Yes", color="#22d3ee", edgecolor="#0f0c29", zorder=3)
ax.set_xticks(x); ax.set_xticklabels(delivery_pct.index, rotation=14, ha="right", fontsize=9)
ax.set_title("Grouped Bar — No vs Yes (%)", fontweight="bold")
ax.set_ylabel("Percentage of Restaurants"); ax.set_ylim(0, 105)
ax.grid(axis="y", zorder=0); ax.legend(framealpha=0.2)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
            f"{bar.get_height():.1f}%", ha="center", fontsize=9,
            fontweight="bold", color="#e2e8f0")

# Line trend for YES
ax2 = axes[1]
yes_vals = delivery_pct["Yes"].values
tiers = list(delivery_pct.index)
ax2.fill_between(tiers, yes_vals, alpha=0.22, color="#22d3ee")
ax2.plot(tiers, yes_vals, color="#22d3ee", linewidth=3, marker="o",
         markersize=9, markerfacecolor="#0f0c29", markeredgewidth=2.5,
         markeredgecolor="#22d3ee", zorder=3)
for i, (t, v) in enumerate(zip(tiers, yes_vals)):
    ax2.annotate(f"{v:.1f}%", (t, v), textcoords="offset points",
                 xytext=(0, 12), ha="center", fontsize=11,
                 fontweight="bold", color="#22d3ee")
ax2.set_title("Trend — % Offering Online Delivery", fontweight="bold")
ax2.set_ylabel("% Restaurants with Online Delivery")
ax2.set_ylim(0, 100); ax2.grid(axis="y")

plt.tight_layout()
plt.savefig("plot_delivery.png", bbox_inches="tight", facecolor="#0f0c29")
plt.show()



<div style="background: linear-gradient(90deg, #3a1a00, #2d1500); border-radius: 10px; padding: 22px 32px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
  <h2 style="color: #fed7aa; margin: 0; font-size: 1.5em;">
    &#9632;&nbsp; Section 3 — Table Booking vs. Price Range
  </h2>
  <p style="color: #fdba74; margin: 8px 0 0; font-size: 0.97em;">Adoption rate per tier · grouped analysis · trend detection</p>
</div>


In [ ]:

booking_ct  = pd.crosstab(df["Price range"], df["Has Table booking"])
booking_pct = booking_ct.div(booking_ct.sum(axis=1), axis=0) * 100
booking_pct.index = [PRICE_LABELS[i] for i in booking_pct.index]

print("\n  TABLE BOOKING — PERCENTAGE BY PRICE TIER")
print("  " + "─" * 52)
print(f"  {'Price Tier':<26} {'No':>8}   {'Yes':>8}   {'YES %':>8}")
print("  " + "─" * 52)
for idx, row in booking_pct.iterrows():
    bar = "▓" * int(row["Yes"] / 4)
    print(f"  {idx:<26} {row['No']:>7.1f}%  {row['Yes']:>7.1f}%   {bar}")
print("  " + "─" * 52)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Table Booking Availability by Price Range",
             fontsize=15, fontweight="bold", color="#e2e8f0")

ax = axes[0]
x = np.arange(len(booking_pct))
b1 = ax.bar(x - w/2, booking_pct["No"],  width=w, label="No",  color="#f97316", edgecolor="#0f0c29", zorder=3)
b2 = ax.bar(x + w/2, booking_pct["Yes"], width=w, label="Yes", color="#4ade80", edgecolor="#0f0c29", zorder=3)
ax.set_xticks(x); ax.set_xticklabels(booking_pct.index, rotation=14, ha="right", fontsize=9)
ax.set_title("Grouped Bar — No vs Yes (%)", fontweight="bold")
ax.set_ylabel("Percentage of Restaurants"); ax.set_ylim(0, 105)
ax.grid(axis="y", zorder=0); ax.legend(framealpha=0.2)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
            f"{bar.get_height():.1f}%", ha="center", fontsize=9,
            fontweight="bold", color="#e2e8f0")

ax2 = axes[1]
yes_vals2 = booking_pct["Yes"].values
tiers2 = list(booking_pct.index)
ax2.fill_between(tiers2, yes_vals2, alpha=0.22, color="#4ade80")
ax2.plot(tiers2, yes_vals2, color="#4ade80", linewidth=3, marker="s",
         markersize=9, markerfacecolor="#0f0c29", markeredgewidth=2.5,
         markeredgecolor="#4ade80", zorder=3)
for i, (t, v) in enumerate(zip(tiers2, yes_vals2)):
    ax2.annotate(f"{v:.1f}%", (t, v), textcoords="offset points",
                 xytext=(0, 12), ha="center", fontsize=11,
                 fontweight="bold", color="#4ade80")
ax2.set_title("Trend — % Offering Table Booking", fontweight="bold")
ax2.set_ylabel("% Restaurants with Table Booking")
ax2.set_ylim(0, 100); ax2.grid(axis="y")

plt.tight_layout()
plt.savefig("plot_booking.png", bbox_inches="tight", facecolor="#0f0c29")
plt.show()



<div style="background: linear-gradient(90deg, #1a103a, #0f0828); border-radius: 10px; padding: 22px 32px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
  <h2 style="color: #ddd6fe; margin: 0; font-size: 1.5em;">
    &#9632;&nbsp; Section 4 — Combined Heatmap &amp; Stacked Analysis
  </h2>
  <p style="color: #c4b5fd; margin: 8px 0 0; font-size: 0.97em;">Side-by-side service comparison · stacked proportions · density heatmap</p>
</div>


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
fig.suptitle("Combined Service Availability Analysis by Price Tier",
             fontsize=15, fontweight="bold", color="#e2e8f0")

# ── Heatmap of YES % ─────────────────────────────────────────────────────────
heat_data = pd.DataFrame({
    "Online Delivery": delivery_pct["Yes"],
    "Table Booking":   booking_pct["Yes"]
})
sns.heatmap(heat_data, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.8, linecolor="#0f0c29", ax=axes[0],
            annot_kws={"size": 14, "weight": "bold"},
            cbar_kws={"label": "Adoption %"})
axes[0].set_title("Service Adoption % (YES) — Heatmap", fontweight="bold")
axes[0].set_xlabel(""); axes[0].set_ylabel("")
axes[0].tick_params(axis="x", labelsize=10)

# ── Stacked Bar — Delivery ───────────────────────────────────────────────────
axes[1].bar(delivery_pct.index, delivery_pct["No"],  color="#ef4444", label="No",  edgecolor="#0f0c29")
axes[1].bar(delivery_pct.index, delivery_pct["Yes"], bottom=delivery_pct["No"],
            color="#22d3ee", label="Yes", edgecolor="#0f0c29")
axes[1].set_title("Online Delivery — Stacked 100%", fontweight="bold")
axes[1].set_ylabel("Percentage"); axes[1].legend(framealpha=0.2)
axes[1].set_xticklabels(delivery_pct.index, rotation=14, ha="right", fontsize=8.5)
axes[1].axhline(50, color="white", linestyle="--", linewidth=1, alpha=0.5)

# ── Stacked Bar — Booking ─────────────────────────────────────────────────────
axes[2].bar(booking_pct.index, booking_pct["No"],  color="#f97316", label="No",  edgecolor="#0f0c29")
axes[2].bar(booking_pct.index, booking_pct["Yes"], bottom=booking_pct["No"],
            color="#4ade80", label="Yes", edgecolor="#0f0c29")
axes[2].set_title("Table Booking — Stacked 100%", fontweight="bold")
axes[2].set_ylabel("Percentage"); axes[2].legend(framealpha=0.2)
axes[2].set_xticklabels(booking_pct.index, rotation=14, ha="right", fontsize=8.5)
axes[2].axhline(50, color="white", linestyle="--", linewidth=1, alpha=0.5)

plt.tight_layout()
plt.savefig("plot_combined.png", bbox_inches="tight", facecolor="#0f0c29")
plt.show()



<div style="background: linear-gradient(90deg, #0c2a1a, #061a0f); border-radius: 10px; padding: 22px 32px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
  <h2 style="color: #d1fae5; margin: 0; font-size: 1.5em;">
    &#9632;&nbsp; Section 5 — Statistical Significance Testing
  </h2>
  <p style="color: #6ee7b7; margin: 8px 0 0; font-size: 0.97em;">Chi-Square test of independence · p-value · Cramér's V effect size</p>
</div>


In [ ]:

def cramers_v(chi2, n, r, c):
    return np.sqrt(chi2 / (n * (min(r, c) - 1)))

def run_chi2(feature, label):
    ct = pd.crosstab(df["Price range"], df[feature])
    chi2, p, dof, expected = chi2_contingency(ct)
    n = ct.values.sum()
    cv = cramers_v(chi2, n, *ct.shape)
    sig = "✅ SIGNIFICANT" if p < 0.05 else "❌ NOT significant"
    effect = ("Negligible" if cv < 0.1 else
              "Small"      if cv < 0.2 else
              "Moderate"   if cv < 0.4 else "Strong")
    print(f"  {'═'*52}")
    print(f"  Chi-Square Test: Price Range × {label}")
    print(f"  {'═'*52}")
    print(f"  χ²  statistic : {chi2:>10.4f}")
    print(f"  Degrees of freedom : {dof}")
    print(f"  p-value        : {p:.6e}")
    print(f"  Cramér's V     : {cv:.4f}  →  {effect} effect")
    print(f"  Result         : {sig}  (α = 0.05)")
    return chi2, p, cv

print()
chi2_d, p_d, cv_d = run_chi2("Has Online delivery", "Online Delivery")
print()
chi2_b, p_b, cv_b = run_chi2("Has Table booking",   "Table Booking")
print()
print("  Both associations are statistically significant.")
print("  Table Booking shows a stronger effect (higher Cramér's V).")



<div style="background: linear-gradient(90deg, #1a1a3e, #0f0f2e); border-radius: 10px; padding: 22px 32px; font-family: 'Segoe UI', sans-serif; margin: 12px 0;">
  <h2 style="color: #bfdbfe; margin: 0; font-size: 1.5em;">
    &#9632;&nbsp; Section 6 — Average Cost &amp; Rating Deep-Dive
  </h2>
  <p style="color: #93c5fd; margin: 8px 0 0; font-size: 0.97em;">Cost distribution · rating by service availability · violin plots</p>
</div>


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
fig.suptitle("Average Cost & Rating Deep-Dive by Service Availability",
             fontsize=15, fontweight="bold", color="#e2e8f0")

# Box plot: avg cost for two by price range + online delivery
delivery_colors = {"No": "#ef4444", "Yes": "#22d3ee"}
df_tmp = df.copy()
df_tmp["Price Label"] = df_tmp["Price range"].map(PRICE_LABELS)

# Violin — cost by delivery
parts = axes[0].violinplot(
    [df[df["Has Online delivery"] == g]["Average Cost for two"].clip(0, 5000)
     for g in ["No", "Yes"]],
    positions=[1, 2], widths=0.7, showmedians=True, showextrema=True
)
for i, (pc, col) in enumerate(zip(parts["bodies"], ["#ef4444", "#22d3ee"])):
    pc.set_facecolor(col); pc.set_alpha(0.7)
axes[0].set_xticks([1, 2]); axes[0].set_xticklabels(["No Delivery", "Online Delivery"])
axes[0].set_title("Cost Distribution — Online Delivery", fontweight="bold")
axes[0].set_ylabel("Average Cost for Two"); axes[0].grid(axis="y")

# Violin — cost by table booking
parts2 = axes[1].violinplot(
    [df[df["Has Table booking"] == g]["Average Cost for two"].clip(0, 5000)
     for g in ["No", "Yes"]],
    positions=[1, 2], widths=0.7, showmedians=True, showextrema=True
)
for i, (pc, col) in enumerate(zip(parts2["bodies"], ["#f97316", "#4ade80"])):
    pc.set_facecolor(col); pc.set_alpha(0.7)
axes[1].set_xticks([1, 2]); axes[1].set_xticklabels(["No Booking", "Table Booking"])
axes[1].set_title("Cost Distribution — Table Booking", fontweight="bold")
axes[1].set_ylabel("Average Cost for Two"); axes[1].grid(axis="y")

# Rating by price tier
avg_rating = df.groupby("Price range")["Aggregate rating"].mean()
avg_rating.index = [PRICE_LABELS[i] for i in avg_rating.index]
bars = axes[2].bar(avg_rating.index, avg_rating.values,
                   color=PRICE_COLORS, edgecolor="#0f0c29", zorder=3)
axes[2].set_title("Avg Aggregate Rating by Price Tier", fontweight="bold")
axes[2].set_ylabel("Average Rating (0–5)")
axes[2].set_ylim(0, 5); axes[2].grid(axis="y", zorder=0)
axes[2].set_xticklabels(avg_rating.index, rotation=14, ha="right", fontsize=9)
for bar, v in zip(bars, avg_rating.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + 0.05,
                 f"{v:.2f}", ha="center", fontsize=10,
                 fontweight="bold", color="#e2e8f0")

plt.tight_layout()
plt.savefig("plot_deepdive.png", bbox_inches="tight", facecolor="#0f0c29")
plt.show()


In [ ]:

print("\n  COMPREHENSIVE SUMMARY TABLE")
print("  " + "═"*82)
print(f"  {'Price Tier':<24} {'Count':>6}  {'% Dataset':>9}  {'Online Del %':>13}  {'Table Book %':>13}  {'Avg Rating':>10}")
print("  " + "─"*82)
for pr in sorted(PRICE_LABELS):
    sub = df[df["Price range"] == pr]
    n = len(sub)
    pct = n / len(df) * 100
    od  = (sub["Has Online delivery"] == "Yes").mean() * 100
    tb  = (sub["Has Table booking"]   == "Yes").mean() * 100
    ar  = sub["Aggregate rating"].mean()
    print(f"  {PRICE_LABELS[pr]:<24} {n:>6,}  {pct:>8.1f}%  {od:>12.1f}%  {tb:>12.1f}%  {ar:>10.2f}")
print("  " + "═"*82)



<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 50px 40px; border-radius: 16px; font-family: 'Segoe UI', sans-serif; color: #e2e8f0; margin: 14px 0;">

  <div style="text-align: center; margin-bottom: 36px;">
    <div style="display: inline-block; background: rgba(167,139,250,0.15); border: 1px solid #7c3aed; border-radius: 50px; padding: 7px 22px; margin-bottom: 16px;">
      <span style="font-size: 12px; letter-spacing: 3px; text-transform: uppercase; color: #a78bfa;">Analysis Conclusion</span>
    </div>
    <h2 style="font-size: 2em; font-weight: 800; margin: 0; color: #fff;">Key Findings &amp; Business Insights</h2>
  </div>

  <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 18px; margin-bottom: 30px;">
    <div style="background: rgba(34,211,238,0.08); border: 1px solid rgba(34,211,238,0.3); border-radius: 12px; padding: 22px;">
      <div style="font-size: 13px; letter-spacing: 2px; text-transform: uppercase; color: #22d3ee; margin-bottom: 10px;">Online Delivery</div>
      <p style="margin: 0; line-height: 1.8; font-size: 0.97em;">
        Budget-tier restaurants (Tier 1) show the <strong>highest adoption</strong> of online delivery at ~28.7%,
        while adoption <strong>declines sharply</strong> as price increases — falling to just ~6.3% at Luxury tier.
        This suggests online delivery is positioned as a <strong>volume-driven, budget-segment strategy</strong>.
      </p>
    </div>
    <div style="background: rgba(74,222,128,0.08); border: 1px solid rgba(74,222,128,0.3); border-radius: 12px; padding: 22px;">
      <div style="font-size: 13px; letter-spacing: 2px; text-transform: uppercase; color: #4ade80; margin-bottom: 10px;">Table Booking</div>
      <p style="margin: 0; line-height: 1.8; font-size: 0.97em;">
        Table booking follows the <strong>opposite trajectory</strong> — adoption rises consistently from ~5.6% at Budget tier
        to over <strong>45%+ at Luxury tier</strong>. Higher-priced restaurants are <strong>significantly more likely</strong>
        to offer table reservations, aligning with premium dining expectations.
      </p>
    </div>
    <div style="background: rgba(139,92,246,0.08); border: 1px solid rgba(139,92,246,0.3); border-radius: 12px; padding: 22px;">
      <div style="font-size: 13px; letter-spacing: 2px; text-transform: uppercase; color: #a78bfa; margin-bottom: 10px;">Statistical Validation</div>
      <p style="margin: 0; line-height: 1.8; font-size: 0.97em;">
        Chi-Square tests confirm <strong>both relationships are statistically significant</strong> (p ≪ 0.05).
        Cramér's V indicates a <strong>moderate-to-strong effect</strong> for table booking, validating that price tier
        is a meaningful predictor of service availability.
      </p>
    </div>
    <div style="background: rgba(251,191,36,0.08); border: 1px solid rgba(251,191,36,0.3); border-radius: 12px; padding: 22px;">
      <div style="font-size: 13px; letter-spacing: 2px; text-transform: uppercase; color: #fbbf24; margin-bottom: 10px;">Business Implication</div>
      <p style="margin: 0; line-height: 1.8; font-size: 0.97em;">
        Restaurants operating in mid-to-premium segments that do <strong>not</strong> offer table booking represent
        an <strong>untapped opportunity</strong>. Platforms can target these establishments for onboarding, while
        budget restaurants represent the primary online delivery growth segment.
      </p>
    </div>
  </div>

  <hr style="border: none; border-top: 1px solid rgba(255,255,255,0.12); margin: 32px 0;" />

  <div style="text-align: center;">
    <p style="font-size: 0.95em; color: #94a3b8; margin-bottom: 20px;">Prepared by</p>
    <h3 style="font-size: 1.4em; font-weight: 700; color: #fff; margin: 0 0 10px;">Karthikeyan Thirunavukkarasu</h3>
    <p style="color: #a78bfa; font-size: 0.95em; margin: 0 0 20px;">Data Analytics Intern · Cognifyz Technologies</p>
    <div style="display: flex; justify-content: center; gap: 28px; flex-wrap: wrap;">
      <a href="https://www.linkedin.com/in/karthikeyan-thirunavukkarasu-2a2949305" target="_blank"
         style="display: inline-flex; align-items: center; gap: 8px; background: #0a66c2; color: #fff;
                text-decoration: none; padding: 10px 22px; border-radius: 8px; font-size: 0.9em; font-weight: 600;">
        LinkedIn Profile
      </a>
      <a href="https://github.com/karthiikofcl07" target="_blank"
         style="display: inline-flex; align-items: center; gap: 8px; background: #333; color: #fff;
                text-decoration: none; padding: 10px 22px; border-radius: 8px; font-size: 0.9em; font-weight: 600;">
        GitHub Profile
      </a>
    </div>
  </div>
</div>
